In [ ]:
from typo_utils import make_typo, ngram_overlap, evaluate_ngram_overlap_mecab, compute_micro_average, ipadic_files
from llama_utils import get_llama_server_models
import MeCab
import pandas as pd
from tqdm.notebook import tqdm
import typo_utils

In [ ]:
import re

def add_units_height_weight(text: str) -> str:
    """
    長文中の [身長・体重] h w を
    [身長・体重] hcm wkg に変換する
    """
    pattern = re.compile(
        r"(\[身長・体重\])"      # ラベル
        r"\s*"                   # 空白
        r"(\d+(?:\.\d+)?)"       # 身長
        r"[ 　]+"                # 半角 or 全角スペース
        r"(\d+(?:\.\d+)?)"       # 体重
    )

    def repl(m):
        label, height, weight = m.groups()
        return f"{label} {height}cm　{weight}kg"

    return pattern.sub(repl, text)

In [ ]:
import json
import re
import requests

def strip_gpt_oss_channels(s: str) -> str:
    """llama-server(gpt-oss)がcontentに混ぜる <|channel|>... を除去して本文だけ返す"""
    if not s:
        return ""

    # final チャンネルがある場合は final だけ採用
    m = re.search(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|$)", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    # analysis + <|end|> + 本文 の形式なら <|end|> 以降だけ採用
    if "<|end|>" in s:
        return s.split("<|end|>")[-1].strip()

    # それ以外：タグを雑に除去
    s = re.sub(r"<\|[^>]*\|>", "", s)
    return s.strip()

def extract_last_report(text: str) -> str:
    """テンプレが2回出る等の事故に備え、最後のヘッダ以降だけ採用"""
    if not text:
        return ""
    header = "【乳癌取扱い規約・第19版.】"
    idxs = [m.start() for m in re.finditer(re.escape(header), text)]
    if not idxs:
        return text.strip()
    return text[idxs[-1]:].strip()

import json
import requests


def explain_with_llama_server_chat(
    selected: dict,
    server_url: str = "http://127.0.0.1:8080/v1/chat/completions",
) -> str:
    system_prompt = """あなたは病理診断レポートの解説作成者です。対象読者は、医師免許取得後数年以内の初期研修医です。
専門性は保ちつつ、「何が分かって」「何が診断の決め手で」「何が臨床的に重要か」が短時間で理解できる解説を行ってください。

【厳守】
- 出力は本文のみ。前置き、挨拶、注釈、自己言及、思考過程、Markdown、箇条書き、英語を出力しない
- 入力JSONに含まれる情報のみを使用し、推測や断定はしない
- 不明・未評価の項目は「不明」と明記する
- 全体は簡潔にし、各項目3〜6文程度を目安とする
- 教科書的な一般論や詳細すぎる分子機序の説明は避ける
- 免疫染色（IHC）については、結果の羅列ではなく「診断上の意味」を必ず説明する
- 治療方針の指示や推奨は行わず、病理学的含意の説明に留める
- 次の出力形式（順序・改行）を厳守し、行の追加や削除をしない

【出力形式（この順序・改行を厳守）】
免疫染色の解釈: 
鑑別診断との関係: 
現時点で不明な点・他に確認すべき事項など: 

【書き方の指針】
- 「免疫染色の解釈」では、各マーカーについて「何を確認する目的で行われ」「その結果がどの診断や臓器由来を支持するか」を簡潔に述べる
- 「鑑別診断との関係」では、形態所見と免疫染色を合わせて、どの鑑別が支持的／否定的かを述べる（断定しすぎない）
- 「現時点で不明な点・他に確認すべき事項など」では、評価不能・未記載だが解釈に重要な項目を列挙する。他に確認すべき所見、免疫染色、検査などがあれば記載する。"""

    # 出力フォーマット固定（モデルに「この6行だけ」を守らせる）
    output_format = """免疫染色の解釈: 
鑑別診断との関係: 
現時点で不明な点・他に確認すべき事項など:  """

    user_prompt = f"""次のJSONに含まれる情報だけを使って、下の出力形式を厳守して解説文を作成してください。
出力形式（このままの順序・改行で出力）：
{output_format}

JSON:
{json.dumps(selected, ensure_ascii=False, indent=2)}
"""

    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        #"temperature": 0.1,
        #"top_p": 1.0,
        #"presence_penalty": 0.0,
        #"max_tokens": 4096,
    }

    try:
        r = requests.post(
            server_url,
            json=payload,
            proxies={"http": None, "https": None},
            timeout=300,
        )
        r.raise_for_status()
        data = r.json()

        content = (data.get("choices", [{}])[0].get("message", {}) or {}).get("content", "")
        content = content.strip()

        # 既存実装がある前提（なければ削除してください）
        try:
            content = strip_gpt_oss_channels(content)  # type: ignore
        except Exception:
            pass

        return content

    except requests.exceptions.RequestException as e:
        print("❌ LLM接続エラー:", e)
        try:
            print("response text:", r.text)  # type: ignore
        except Exception:
            pass
        return ""

In [ ]:
dic_dir = "/Users/mkawai/Developers/llm/patholgical_report/mecab-ipadic-2.7.0-20070610"
usr_dir = "/Users/mkawai/Developers/llm/patholgical_report/user.dic"

tagger = MeCab.Tagger(f'-r /dev/null -d "{dic_dir}" -u "{usr_dir}"')

df_path = "./reports.tsv"
df = pd.read_csv(df_path, sep='\t')


In [ ]:
import time
results_mecab_with_counts = []
results_char_with_counts = []
#random.shuffle(index)
llama_times = []

if '解説' not in df.columns:
    df['解説'] = ""

if '入力' not in df.columns:
    df['入力'] = ""

if 'model_name' not in df.columns:
    df['model_name'] = ""

for progress, i in tqdm(enumerate(range(len(df)) , start=0), total=len(df) ):
    model_name = get_llama_server_models("http://localhost:8080")["data"][0]["id"]
    df.at[i, 'model_name'] = model_name
    selected = df.iloc[i]
    original = f"""{selected['患者年齢(年)_x']}歳、{selected['性別名_x']}。{add_units_height_weight(selected['臨床所見'])}
     病理診断：{selected['診断名（最新）_x']}
     病理所見：{selected['所見_改変']}
    """
    original = typo_utils.normalize_unicode_symbol_white( original )
    t0 = time.perf_counter()
    fixed = explain_with_llama_server_chat(original, server_url="http://localhost:8080/v1/chat/completions")
    t1 = time.perf_counter()
    llama_times.append(t1 - t0)

    fixed = typo_utils.normalize_unicode_symbol_white(fixed)
    
    df.at[i, '入力'] = original
    df.at[i, '解説'] = fixed

    print(fixed)

In [ ]:
import numpy as np
print(f"llama中央値応答時間: {np.median(llama_times):.3f} 秒")
print(f"llama平均応答時間: {np.mean(llama_times):.3f} 秒")
print(f"llama最小応答時間: {np.min(llama_times):.3f} 秒")
print(f"llama最大応答時間: {np.max(llama_times):.3f} 秒")


df['llama中央値応答時間'] = np.median(llama_times)
df['llama平均応答時間'] = np.mean(llama_times)
df['llama最小応答時間'] = np.min(llama_times)
df['llama最大応答時間'] = np.max(llama_times)

In [ ]:
from pathlib import Path
cols = ["model_name", "入力", "解説", 'llama中央値応答時間', 'llama平均応答時間', 'llama最小応答時間', 'llama最大応答時間']
df[cols].to_csv(f"output_{Path(model_name).stem}.csv", index=False, encoding="utf-8-sig")